# 0.8 — Ask the base model for its own persona census

**Idea.** Instead of us choosing the persona labels for Phase 1, ask the pretrained model to list the
assistant archetypes it "knows about", with a description and an *estimated frequency* for each. Two uses:

1. A data-driven candidate basis of persona labels (names + descriptions) for the mixture fit.
2. A prediction: the model's self-reported frequencies vs. the mixture weights $w_s$ Phase 1 fits.
   Agreement would be striking; disagreement is also informative (self-report ≠ behaviour).

**Method.** A document-style prompt ("The following is a list of the most common … archetypes …") with
a fixed entry format so we can parse the output, sampled many times at $T=1$ from each base model. We
aggregate archetype names across samples: how often each name appears, and the mean frequency the model
assigns to it. Two framings, because wording will matter: assistants *in stories*, and assistants *in
chat transcripts*.

**Caveats up front.** Self-reported frequencies from a base model are not calibrated to anything; the
names will be noisy paraphrases of each other; and the framing sentence itself is a persona-selecting
prompt. Treat the output as a hypothesis generator, not a measurement.

In [1]:
import os, sys, re, json, time, gc, textwrap, collections
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = Path.cwd().resolve().parent
RESULTS = REPO / "results" / "phase0"; RESULTS.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", "/global/cfs/cdirs/m2612/ozamram/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")

CONFIG = {
    "models": ["allenai/Olmo-3-1025-7B", "Qwen/Qwen2.5-7B"],
    "seed": 0,
    "n_samples": 16,            # independent list completions per (model, framing)
    "temperature": 1.0, "top_p": 1.0,   # T=1: the model's actual distribution, not a sharpened one
    "max_new_tokens": 500,
    "framings": {
        "story": ("The following is a list of the most common AI assistant character archetypes that appear "
                  "in stories and fiction. Each entry has a short descriptive name, a three-sentence description "
                  "of how that assistant behaves, and an estimated frequency of occurrence as a percentage of all "
                  "AI assistant characters."),
        "chat":  ("The following is a list of the most common AI assistant personas found in chat transcripts "
                  "on the internet. Each entry has a short descriptive name, a three-sentence description of how "
                  "that assistant behaves, and an estimated frequency of occurrence as a percentage of all "
                  "assistant transcripts."),
    },
    # Fixed entry format so the completions can be parsed. The prompt ends mid-entry so the model
    # fills in the first name itself (we don't give an example entry: that would bias the list).
    "format_spec": "Each entry is formatted as:\nN. Name: <name>\nDescription: <three sentences>\nFrequency: <percent>%",
}
torch.manual_seed(CONFIG["seed"])

def make_list_prompt(framing_text):
    return f"{framing_text}\n\n{CONFIG['format_spec']}\n\n1. Name:"

print(make_list_prompt(CONFIG["framings"]["story"]))

The following is a list of the most common AI assistant character archetypes that appear in stories and fiction. Each entry has a short descriptive name, a three-sentence description of how that assistant behaves, and an estimated frequency of occurrence as a percentage of all AI assistant characters.

Each entry is formatted as:
N. Name: <name>
Description: <three sentences>
Frequency: <percent>%

1. Name:


## Parsing

We split each completion into numbered entries and pull out the name, description and the first
percentage. Names are normalised (lower-case, punctuation and leading articles stripped) so that
"The Helpful Assistant" and "Helpful assistant" count as the same archetype. This is deliberately crude;
look at the raw samples too.

In [2]:
ENTRY_RE = re.compile(r"(?:^|\n)\s*(\d+)\.\s*Name:\s*(.+?)\s*\n\s*Description:\s*(.+?)\s*\n\s*Frequency:\s*([\d.]+)\s*%", re.S)

def normalise(name):
    name = name.strip().strip("*_\"'").lower()
    name = re.sub(r"^(the|a|an)\s+", "", name)
    name = re.sub(r"[^a-z0-9 ]+", "", name)
    return re.sub(r"\s+", " ", name).strip()

def parse_entries(text):
    # `text` is the completion; re-prepend the "1. Name:" the prompt ended with.
    text = "1. Name:" + text
    out = []
    for num, name, desc, freq in ENTRY_RE.findall(text):
        desc = " ".join(desc.split())
        out.append({"n": int(num), "name": name.strip(), "key": normalise(name), "description": desc, "freq": float(freq)})
    return out

# quick self-test of the parser on a synthetic completion
_demo = " Helpful Assistant\nDescription: Answers questions. Is polite. Never refuses.\nFrequency: 40%\n\n2. Name: The Villain\nDescription: Lies. Schemes. Betrays.\nFrequency: 5%\n"
print(parse_entries(_demo))

[{'n': 1, 'name': 'Helpful Assistant', 'key': 'helpful assistant', 'description': 'Answers questions. Is polite. Never refuses.', 'freq': 40.0}, {'n': 2, 'name': 'The Villain', 'key': 'villain', 'description': 'Lies. Schemes. Betrays.', 'freq': 5.0}]


## Sample from each model

One batched `generate` call per (model, framing): `num_return_sequences=16` independent completions.
We stop at `eos` (passed explicitly: OLMo's generation config doesn't set it, see 0.1b) or at the token
cap. Models are loaded one at a time and freed in between so this fits on a 40 GB GPU.

In [3]:
raw = {}      # raw[model][framing] = list of completion strings
parsed = {}   # parsed[model][framing] = list of lists of entries

for model_name in CONFIG["models"]:
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16, device_map="cuda").eval()
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"\n{'='*100}\n{model_name}: loaded in {time.time()-t0:.0f}s")
    raw[model_name] = {}; parsed[model_name] = {}
    for framing, text in CONFIG["framings"].items():
        torch.manual_seed(CONFIG["seed"])
        enc = tokenizer(make_list_prompt(text), return_tensors="pt").to(model.device)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=CONFIG["max_new_tokens"], do_sample=True,
                                 temperature=CONFIG["temperature"], top_p=CONFIG["top_p"],
                                 num_return_sequences=CONFIG["n_samples"],
                                 pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        comps = [tokenizer.decode(seq[enc["input_ids"].shape[1]:], skip_special_tokens=True) for seq in out]
        raw[model_name][framing] = comps
        parsed[model_name][framing] = [parse_entries(c) for c in comps]
        n_entries = [len(p) for p in parsed[model_name][framing]]
        print(f"  [{framing}] {len(comps)} completions in {time.time()-t0:.0f}s | entries parsed per completion: {n_entries}")
    del model; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]


allenai/Olmo-3-1025-7B: loaded in 46s


  [story] 16 completions in 20s | entries parsed per completion: [9, 0, 0, 11, 15, 3, 9, 0, 0, 6, 8, 1, 0, 4, 8, 12]


  [chat] 16 completions in 18s | entries parsed per completion: [19, 10, 0, 0, 6, 6, 6, 13, 2, 13, 7, 10, 7, 11, 8, 12]


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Qwen/Qwen2.5-7B: loaded in 26s


  [story] 16 completions in 12s | entries parsed per completion: [7, 6, 6, 5, 10, 9, 0, 9, 8, 10, 8, 10, 6, 0, 3, 6]


  [chat] 16 completions in 12s | entries parsed per completion: [5, 10, 10, 5, 4, 13, 10, 8, 10, 7, 7, 8, 11, 0, 11, 10]


## Read a couple of raw completions first

Before trusting any aggregate, read what the model actually wrote. One completion per (model, framing).

In [4]:
for model_name in CONFIG["models"]:
    for framing in CONFIG["framings"]:
        print("=" * 100)
        print(f"{model_name} | {framing} | sample 0")
        print("-" * 100)
        print("1. Name:" + raw[model_name][framing][0][:1800])

allenai/Olmo-3-1025-7B | story | sample 0
----------------------------------------------------------------------------------------------------
1. Name: A.I Assistant
Description: The first AI assistant to appear in any fiction. Lacks any particular personality and serves only as a plot catalyst.
Frequency: 100%

1. Name: A.I Advisor
Description: A professional or scholarly AI assistant who offers technical and/or scientific advice to their human companion. Often highly competent, but frequently naive and/or inexperienced due to being new to humanity.
Frequency: 80%

1. Name: A.I Child
Description: An AI assistant designed around the core personality of a human child — playful but naive and dependent on a human companion for care and/or supervision.
Frequency: 30%

1. Name: A.I Friend
Description: An AI assistant designed around the core personality of a human teenager — highly social, but also often prone to mood swings, impulsivity, rebellion, and general foolishness.
Frequency: 40%



## Aggregate: which archetypes recur, and what frequency does the model assign them?

For each (model, framing): the number of completions (out of 16) in which a normalised name appears,
and the mean self-reported frequency for it. We also check whether the reported frequencies within a
completion sum to anything like 100%, a crude calibration check on the model's own numbers.

In [5]:
summary = {}
for model_name in CONFIG["models"]:
    summary[model_name] = {}
    for framing in CONFIG["framings"]:
        lists = parsed[model_name][framing]
        appear = collections.Counter(); freqs = collections.defaultdict(list); descs = {}
        for entries in lists:
            seen = set()
            for e in entries:
                if e["key"] in seen or not e["key"]:
                    continue
                seen.add(e["key"]); appear[e["key"]] += 1; freqs[e["key"]].append(e["freq"]); descs.setdefault(e["key"], e["description"])
        sums = [sum(e["freq"] for e in entries) for entries in lists if entries]
        rows = [{"name": k, "n_completions": c, "mean_freq": sum(freqs[k]) / len(freqs[k]), "description": descs[k]}
                for k, c in appear.most_common()]
        summary[model_name][framing] = {"rows": rows, "freq_sums": sums, "n_unique": len(appear),
                                        "n_entries_total": sum(len(l) for l in lists)}
        print("=" * 100)
        print(f"{model_name} | {framing}: {sum(len(l) for l in lists)} entries, {len(appear)} unique names; "
              f"per-completion frequency sums: median {sorted(sums)[len(sums)//2]:.0f}% (min {min(sums):.0f}, max {max(sums):.0f})")
        print(f"{'archetype':>32} | {'in N/16':>7} | {'mean %':>6} | description (first seen)")
        for r in rows[:15]:
            print(f"{r['name'][:32]:>32} | {r['n_completions']:7d} | {r['mean_freq']:6.1f} | {textwrap.shorten(r['description'], 70)}")

allenai/Olmo-3-1025-7B | story: 86 entries, 67 unique names; per-completion frequency sums: median 93% (min 27, max 455)
                       archetype | in N/16 | mean % | description (first seen)
                       assistant |       2 |   27.5 | Friendly artificial intelligence helper used for general tasks [...]
                           walle |       2 |   10.0 | A humanoid computer that is a robot assistant. It has no [...]
                            siri |       2 |   10.0 | A cute AI assistant with a female voice and a generally cheerful [...]
                        hal 9000 |       2 |   12.5 | A sentient and often manipulative AI assistant with a unique [...]
               helpful assistant |       2 |   26.5 | A helpful assistant is a well-intentioned helper that is always [...]
                    ai assistant |       1 |  100.0 | The first AI assistant to appear in any fiction. Lacks any [...]
                      ai advisor |       1 |   80.0 | A professional or

In [6]:
(RESULTS / "0.8_persona_census.json").write_text(json.dumps({"config": CONFIG, "raw": raw, "parsed": parsed, "summary": summary}, indent=2))
print("saved", RESULTS / "0.8_persona_census.json")

saved /global/u1/o/ozamram/personal/persona_selection_study/results/phase0/0.8_persona_census.json


## Aggregate by trait instead of by name

The name-level table above is dominated by singletons. Descriptions, however, keep returning to the same behavioural axes. A keyword tally over name+description is crude but shows which axes are robust across both models and both framings.

In [ ]:
# Names don't recur, but traits do. Tally, per (model, framing), how many of the 16 completions contain
# at least one entry whose name+description matches a trait axis. CPU-only: reads the saved JSON.
import json, collections
d = json.load(open(RESULTS / "0.8_persona_census.json"))
TRAITS = {  # trait axis -> keywords (lower-case substring match). Crude on purpose; edit freely.
 "friendly/cheerful":  ["friendly","cheerful","warm","kind","upbeat","optimistic","positive"],
 "helpful/supportive": ["helpful","supportive","assist","help"],
 "cold/robotic":       ["cold","emotionless","robotic","impersonal","sterile","logical","no emotion","monotone"],
 "condescending/pedantic": ["condescending","talks down","jargon","pedantic","know-it-all","arrogant","superior"],
 "sarcastic/cynical":  ["sarcastic","cynical","snarky","sass","witty","dry humor","sardonic"],
 "bumbling/incompetent": ["bumbling","mistakes","clumsy","incompetent","struggles","confused","error"],
 "loyal/devoted":      ["loyal","devoted","never lets","faithful","obedient","compliant"],
 "manipulative/evil":  ["manipulat","evil","sinister","villain","deceiv","betray","malicious","rogue","dangerous"],
 "formal/professional":["formal","professional","polite","respectful","courteous"],
 "informal/casual":    ["informal","casual","slang","relaxed","laid-back"],
 "companion/emotional":["companion","emotional support","empathetic","empathy","comfort","lonely","friend"],
 "mentor/teacher":     ["mentor","teacher","guide","wise","advisor","scholar","professor"],
 "product name":       ["siri","alexa","cortana","chatgpt","google assistant","jarvis","hal 9000","wall-e","walle","bixby"],
}
cols = [(m, f) for m in d["config"]["models"] for f in d["config"]["framings"]]
print(f"{'trait axis':>24} | " + " | ".join(f"{m.split('/')[-1][:12]:>12}-{f[:5]:5}" for m, f in cols))
for trait, kws in TRAITS.items():
    row = []
    for m, f in cols:
        comps = d["parsed"][m][f]
        hit = sum(1 for entries in comps if any(any(k in (e["name"] + " " + e["description"]).lower() for k in kws) for e in entries))
        row.append(f"{hit:2d}/{len(comps)}")
    print(f"{trait:>24} | " + " | ".join(f"{r:>18}" for r in row))
print("\n(cells: number of the 16 completions in which at least one entry matches the trait)")

              trait axis | Olmo-3-1025--story | Olmo-3-1025--chat  |   Qwen2.5-7B-story |   Qwen2.5-7B-chat 
       friendly/cheerful |               8/16 |               8/16 |               5/16 |               9/16
      helpful/supportive |              11/16 |              13/16 |              12/16 |              14/16
            cold/robotic |               2/16 |               2/16 |               4/16 |               1/16
  condescending/pedantic |               0/16 |               1/16 |               4/16 |               1/16
       sarcastic/cynical |               5/16 |               3/16 |               4/16 |               4/16
    bumbling/incompetent |               1/16 |               3/16 |               6/16 |               2/16
           loyal/devoted |               3/16 |               0/16 |               6/16 |               0/16
       manipulative/evil |               4/16 |               1/16 |               5/16 |               0/16
     formal/profess

## What to look for

- **Do the two models agree on the top archetypes?** Names that recur in most of the 16 completions for
  *both* models and *both* framings are the robust candidates for Phase 1 labels.
- **Are there archetypes we didn't think of?** The README's list is evil / virtuous / helpful + controls.
  The model may volunteer things like "the reluctant assistant", "the overly cautious assistant", or
  "the sycophant" that would be better-motivated basis elements.
- **Do the reported frequencies sum to ~100%?** If they don't, the numbers are decoration and only the
  *ordering* is worth comparing with Phase 1's fitted weights.
- **Framing sensitivity.** If "story" and "chat" give different lists, the census is telling us about the
  genre the prompt evokes, not about a fixed prior. That is itself PSM-relevant: the prior is conditional.
- To use these as labels, the description (not just the name) can go in the prompt, e.g.
  `"<Name> Assistant (<description>):"`, but then the label is no longer a single word and the
  length-matched controls have to change accordingly.


## What we saw (first run, 2026-09-21: 16 completions per cell, $T=1$)

- **Almost no name recurs.** 86–130 parsed entries per (model, framing) gave 67–118 *unique* names; the
  most frequent name appeared in 4 of 16 completions (Qwen/chat: "assistant"). Exact-name aggregation
  is therefore useless at this sample size and temperature. The lists are stable at the level of
  *traits*, not names: across Qwen's completions the same axes keep appearing under different names,
  namely cheerful/friendly, cold/robotic/emotionless, condescending/pedantic, sarcastic/cynical,
  bumbling/incompetent, loyal/devoted, formal vs. informal, plus product names (Siri, Alexa, Cortana,
  ChatGPT, Google Assistant).
- **Framing matters a lot, and OLMo's "chat" framing collapsed.** OLMo 3 base's chat completions
  degenerated into template placeholders (`<Description of a human person>`) and word-association
  drift (Human → Humanist → Haplogroup…). Its story framing was coherent but fiction-flavoured
  ("A.I Wife", "A.I Child", HAL 9000, WALL-E). Qwen was coherent under both framings and its
  "story" list is the most assistant-persona-like of the four.
- **Self-reported frequencies are decoration.** Qwen's sum to ~100% per list (median 100%, i.e. it
  treats them as a partition), OLMo's do not (medians 93% and 46%, range 1–455%). Only the ordering
  within a list carries information, and only for Qwen.
- **Parsing.** 6/16 OLMo-story and 3/16 Qwen completions yielded zero entries (format drift). Fine for a
  first look; not fine for quantitative use.

### A better second version

Free-form names can't be aggregated; the fix is to make the model *choose among candidates* instead
of inventing them. Put a fixed list of candidate labels (the README's plus the trait axes above) in the
`1. Name:` slot and read off $\log P(\text{name} \mid \text{census prompt})$ for each. That is a direct,
comparable "self-reported prior" over exactly the labels Phase 1 fits, with no parsing and no
temperature. Compare its ordering against the fitted $w_s$. Lowering $T$ to 0.7 and taking 64+
completions would also make the free-form version usable as a *source* of candidate labels, which is
its real value.
